# Clinic activity D0 — kiểm quyền enforce môi trường

Import vào notebook Kaggle **CPU mới**, rồi **Run All**. Không cần Internet/package hoặc notebook D1/Q6. Phép kiểm này giải quyết quyền cgroup còn thiếu; không chạy lại D1.

Đọc namespace/ancestors và thử tạo subtree UUID mới. Không remount/sudo, sửa controller của parent hiện có hoặc chuyển process có sẵn. Nếu không được delegated CPU/memory thì dừng trước cài/start dịch vụ. Nếu có quyền, tạo budget đề xuất stack2CPU/8GiB, generator1CPU/2GiB; kiểm readback/PID và hai helper nhỏ: CPU0.1CPU có throttling, memory32MiB với helper64MiB bị OOM trong group riêng. Dừng đúng helper do run tạo; giữ empty cgroups, không xóa file/data.

Đây là kiểm capability bằng helper tổng hợp, **không phải full-stack enforcement proof hoặc benchmark**. D0 vẫn PARTIAL tới khi có full trial/process/roles/pools/profile được xác minh. Nếu output CGROUP_READ_ONLY hoặc CPU_MEMORY_NOT_DELEGATED_TO_CHILDREN, kernel đó chưa đáp ứng gate; không cài stack để cố đổi kết quả. ZIP chỉ có JSON metadata/cgroup/PID của helper, không env/context/token/log/DB.


In [ ]:
import base64,hashlib,json,platform,uuid,zipfile
from pathlib import Path
from IPython.display import display,FileLink
assert platform.system()=='Linux' and Path('/kaggle/working').is_dir(), 'Run in the assigned Kaggle CPU notebook.'
PROBE_SHA256='d0c63bd069c8a9b83833b1b2b9f0d3e09260e007b8878fd84c435b783405000f'
PROBE_BASE64='IiIiRDAgY2dyb3VwIGNhcGFiaWxpdHkvZW5mb3JjZW1lbnQgcHJvYmU7IG5vIGFwcGxpY2F0aW9uL0RCL2Jyb2tlciBzdGFydHVwLgoKT25seSBjcmVhdGVzIGEgVVVJRC1vd25lZCBzdWJ0cmVlIGJlbG93IHRoZSBjb2xsZWN0b3IncyB2ZXJpZmllZCBjZ3JvdXAuIE5ldmVyCndyaXRlcyB0aGUgZXhpc3RpbmcgcGFyZW50IGNvbnRyb2xzLCBtaWdyYXRlcyBleGlzdGluZyBQSURzLCByZW1vdW50cyBvciBzdWRvZXMuClN5bnRoZXRpYyBwcmVzc3VyZSBpcyBjb25maW5lZCB0byBvd25lZCBoZWxwZXIgY2dyb3Vwcy4gRW1wdHkgZ3JvdXBzIGFyZSBrZXB0LgoiIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQppbXBvcnQgZXJybm8KaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmUKaW1wb3J0IHNpZ25hbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB1dWlkCgpCVURHRVQgPSB7J3N0YXR1cyc6ICdQUk9QT1NFRF9OT1RfRlJPWkVOJywgJ3N0YWNrJzogeydjcHVNYXgnOiAnMjAwMDAwIDEwMDAwMCcsICdtZW1vcnlCeXRlcyc6IDgqMTAyNCoqM30sCiAgICAgICAgICAnZ2VuZXJhdG9yJzogeydjcHVNYXgnOiAnMTAwMDAwIDEwMDAwMCcsICdtZW1vcnlCeXRlcyc6IDIqMTAyNCoqM30sCiAgICAgICAgICAnc3dhcE1heEJ5dGVzJzogMCwgJ2lvTGltaXQnOiBOb25lfQpDT05UUk9MUyA9ICgnY3B1Lm1heCcsICdjcHUuc3RhdCcsICdtZW1vcnkubWF4JywgJ21lbW9yeS5jdXJyZW50JywgJ21lbW9yeS5zd2FwLm1heCcsCiAgICAgICAgICAgICdtZW1vcnkuZXZlbnRzJywgJ2NwdXNldC5jcHVzLmVmZmVjdGl2ZScsICdpby5tYXgnLCAnY2dyb3VwLmNvbnRyb2xsZXJzJywgJ2Nncm91cC5zdWJ0cmVlX2NvbnRyb2wnKQoKCmNsYXNzIFVuYXZhaWxhYmxlKFJ1bnRpbWVFcnJvcik6CiAgICBwYXNzCgoKZGVmIHVuZXNjYXBlKHZhbHVlKToKICAgIHJldHVybiByZS5zdWIocidcXChbMC03XXszfSknLCBsYW1iZGEgbTogY2hyKGludChtWzFdLCA4KSksIHZhbHVlKQoKCmRlZiBsb2NhdGlvbnMobW91bnRpbmZvLCBtZW1iZXJzaGlwKToKICAgICIiIlB1cmUgbmFtZXNwYWNlLWF3YXJlIG1hcHBpbmcsIGluY2x1ZGluZyBhIGNncm91cCBuYW1lc3BhY2Ugcm9vdCAnLycuIiIiCiAgICBtZW1iZXJzID0gW3guc3BsaXQoJzonLCAyKVsyXSBmb3IgeCBpbiBtZW1iZXJzaGlwLnNwbGl0bGluZXMoKSBpZiB4LnN0YXJ0c3dpdGgoJzA6OicpXQogICAgaWYgbGVuKG1lbWJlcnMpICE9IDEgb3Igbm90IG1lbWJlcnNbMF0uc3RhcnRzd2l0aCgnLycpIG9yICcuLicgaW4gUHVyZVBvc2l4UGF0aChtZW1iZXJzWzBdKS5wYXJ0czoKICAgICAgICByYWlzZSBVbmF2YWlsYWJsZSgnQ0dST1VQX1YyX01FTUJFUlNISVBfVU5SRVNPTFZFRCcpCiAgICBtZW1iZXIgPSBtZW1iZXJzWzBdOyByZXN1bHQgPSBbXQogICAgZm9yIGxpbmUgaW4gbW91bnRpbmZvLnNwbGl0bGluZXMoKToKICAgICAgICBsZWZ0LCBzZXAsIHJpZ2h0ID0gbGluZS5wYXJ0aXRpb24oJyAtICcpOyBmaWVsZHMgPSBsZWZ0LnNwbGl0KCk7IGZzID0gcmlnaHQuc3BsaXQoKQogICAgICAgIGlmIG5vdCBzZXAgb3IgbGVuKGZpZWxkcykgPCA2IG9yIGxlbihmcykgPCAzIG9yIGZzWzBdICE9ICdjZ3JvdXAyJzoKICAgICAgICAgICAgY29udGludWUKICAgICAgICByb290LCBtb3VudCA9IHVuZXNjYXBlKGZpZWxkc1szXSksIHVuZXNjYXBlKGZpZWxkc1s0XSkKICAgICAgICBpZiBub3Qgcm9vdC5zdGFydHN3aXRoKCcvJykgb3Igbm90IG1vdW50LnN0YXJ0c3dpdGgoJy8nKSBvciAnLi4nIGluIFB1cmVQb3NpeFBhdGgocm9vdCkucGFydHMgb3IgJy4uJyBpbiBQdXJlUG9zaXhQYXRoKG1vdW50KS5wYXJ0czoKICAgICAgICAgICAgcmFpc2UgVW5hdmFpbGFibGUoJ0NHUk9VUF9NT1VOVF9QQVRIX1VOU0FGRScpCiAgICAgICAgc3VmZml4ID0gbWVtYmVyW2xlbihyb290KTpdIGlmIHJvb3QgIT0gJy8nIGFuZCAobWVtYmVyID09IHJvb3Qgb3IgbWVtYmVyLnN0YXJ0c3dpdGgocm9vdCsnLycpKSBlbHNlIG1lbWJlcgogICAgICAgIGN1cnJlbnQgPSBzdHIoUHVyZVBvc2l4UGF0aChtb3VudCkgLyBzdWZmaXgubHN0cmlwKCcvJykpCiAgICAgICAgcmVzdWx0LmFwcGVuZCh7J21vdW50JzogbW91bnQsICdtb3VudFJvb3QnOiByb290LCAnY3VycmVudCc6IGN1cnJlbnQsCiAgICAgICAgICAgICAgICAgICAgICAgJ2ZsYWdzJzogc29ydGVkKHNldChmaWVsZHNbNV0uc3BsaXQoJywnKSArIGZzWzJdLnNwbGl0KCcsJykpKX0pCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIHJlYWQocGF0aCk6CiAgICB0cnk6CiAgICAgICAgdmFsdWUgPSBQYXRoKHBhdGgpLnJlYWRfdGV4dCgpLnN0cmlwKCkKICAgICAgICBpZiBsZW4odmFsdWUpID4gMTYzODQ6IHJhaXNlIFVuYXZhaWxhYmxlKCdDT05UUk9MX09VVFBVVF9UT09fTEFSR0UnKQogICAgICAgIHJldHVybiB2YWx1ZQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXJyb3I6CiAgICAgICAgcmV0dXJuIHsnZXJybm8nOiBlcnJvci5lcnJub30KCgpkZWYgY291bnRlcnMocGF0aCk6CiAgICB2YWx1ZSA9IHJlYWQocGF0aCkKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBzdHIpOiByYWlzZSBVbmF2YWlsYWJsZSgnQ09VTlRFUlNfVU5SRUFEQUJMRScpCiAgICByZXR1cm4ge2s6IGludCh2KSBmb3IgaywgdiBpbiAobGluZS5zcGxpdCgpIGZvciBsaW5lIGluIHZhbHVlLnNwbGl0bGluZXMoKSl9CgoKZGVmIGJ1ZGdldF9maXRzKGFuY2VzdG9ycywgYWZmaW5pdHkpOgogICAgY3B1ID0gW2xlbihhZmZpbml0eSldOyBtZW1vcnkgPSBbXQogICAgZm9yIHJvdyBpbiBhbmNlc3RvcnM6CiAgICAgICAgcmF3ID0gcm93Wydjb250cm9scyddWydjcHUubWF4J10KICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyYXcsIHN0cik6IHJhaXNlIFVuYXZhaWxhYmxlKCdBTkNFU1RPUl9DUFVfTElNSVRfVU5SRUFEQUJMRScpCiAgICAgICAgcXVvdGEsIHBlcmlvZCA9IHJhdy5zcGxpdCgpCiAgICAgICAgaWYgcXVvdGEgIT0gJ21heCc6IGNwdS5hcHBlbmQoaW50KHF1b3RhKS9pbnQocGVyaW9kKSkKICAgICAgICByYXcgPSByb3dbJ2NvbnRyb2xzJ11bJ21lbW9yeS5tYXgnXQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJhdywgc3RyKTogcmFpc2UgVW5hdmFpbGFibGUoJ0FOQ0VTVE9SX01FTU9SWV9MSU1JVF9VTlJFQURBQkxFJykKICAgICAgICBpZiByYXcgIT0gJ21heCc6IG1lbW9yeS5hcHBlbmQoaW50KHJhdykpCiAgICBpZiBub3QgbWVtb3J5OiByYWlzZSBVbmF2YWlsYWJsZSgnRklOSVRFX0FOQ0VTVE9SX01FTU9SWV9FTlZFTE9QRV9OT1RfUFJPVkVOJykKICAgIGlmIG1pbihjcHUpIDwgMyBvciBtaW4obWVtb3J5KSA8IDEwKjEwMjQqKjM6CiAgICAgICAgcmFpc2UgVW5hdmFpbGFibGUoJ1BST1BPU0VEX0JVREdFVF9FWENFRURTX09CU0VSVkVEX0VOVkVMT1BFJykKICAgIHJldHVybiB7J2VmZmVjdGl2ZUNwdVVwcGVyQm91bmQnOiBtaW4oY3B1KSwgJ21lbW9yeVVwcGVyQm91bmRCeXRlcyc6IG1pbihtZW1vcnkpLAogICAgICAgICAgICAnaXNSZXNlcnZhdGlvbk9yQ29UZW5hbnRJc29sYXRpb25Qcm9vZic6IEZhbHNlfQoKCkhFTFBFUiA9ICcnJ2ltcG9ydCBzeXMsdGltZSxqc29uCmtpbmQ9c3lzLmFyZ3ZbMV0KaWYgc3lzLnN0ZGluLmJ1ZmZlci5yZWFkbGluZSgpIT1iJ3J1blxcbic6c3lzLmV4aXQoMikKaWYga2luZD09J2NwdSc6CiAgICBzdGFydD10aW1lLnByb2Nlc3NfdGltZSgpCiAgICB3aGlsZSB0aW1lLnByb2Nlc3NfdGltZSgpLXN0YXJ0PDAuMjpwYXNzCmVsaWYga2luZD09J21lbW9yeSc6CiAgICBkYXRhPWJ5dGVhcnJheSg2NCoxMDI0KjEwMjQpCiAgICBmb3IgaSBpbiByYW5nZSgwLGxlbihkYXRhKSw0MDk2KTpkYXRhW2ldPTEKZWxzZToKICAgIHByaW50KGpzb24uZHVtcHMoeydtZW1iZXJzaGlwJzpvcGVuKCcvcHJvYy9zZWxmL2Nncm91cCcpLnJlYWQoKS5zdHJpcCgpfSkpCicnJwoKCmRlZiBydW5fcHJvYmUoYnVkZ2V0PU5vbmUpOgogICAgc2VsZWN0ZWQgPSBqc29uLmxvYWRzKGpzb24uZHVtcHMoQlVER0VUIGlmIGJ1ZGdldCBpcyBOb25lIGVsc2UgYnVkZ2V0KSkKICAgICMgTm8gcGFyYW1ldGVyIGNhbiBpbmNyZWFzZSB0aGUgcHJvcG9zZWQgYWxsb2NhdGlvbiBvciBwcmVzc3VyZSBwcm9iZSBzaXplcy4KICAgIGlmIHNlbGVjdGVkICE9IEJVREdFVDogcmFpc2UgVmFsdWVFcnJvcignVU5SRUdJU1RFUkVEX0VOVklST05NRU5UX0JVREdFVCcpCiAgICBydW4gPSBzdHIodXVpZC51dWlkNCgpKTsgb3duZWQgPSBbXTsgcHJvY2Vzc2VzID0gW10KICAgIHJlcG9ydCA9IHsnZm9ybWF0JzogJ2NsaW5pYy1hY3Rpdml0eS1kMC1lbmZvcmNlbWVudC1wcm9iZS12MScsICdydW4nOiBydW4sCiAgICAgICAgICAgICAgJ2NhcHR1cmVkQXRVdGMnOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKSwgJ3N0YXR1cyc6ICdTVEFSVElORycsCiAgICAgICAgICAgICAgJ2QwJzogJ1BBUlRJQUxfRU5WSVJPTk1FTlQnLCAncGVyZm9ybWFuY2UnOiAnTk9UX1JVTicsICdidWRnZXRQcm9wb3NhbCc6IHNlbGVjdGVkLAogICAgICAgICAgICAgICdhcHBsaWNhdGlvblN0YXJ0ZWQnOiBGYWxzZSwgJ2RhdGFiYXNlQ29ubmVjdGlvbnMnOiAwLCAnZXhpc3RpbmdQYXJlbnRDb250cm9sc01vZGlmaWVkJzogRmFsc2UsCiAgICAgICAgICAgICAgJ3Jlc291cmNlRW5mb3JjZW1lbnQnOiAnTk9UX1BST1ZFTicsICdmdWxsU3RhY2tQaWRSb2xlUG9vbENvdmVyYWdlJzogJ05PVF9SVU4nLAogICAgICAgICAgICAgICdvd25lZENncm91cHNSZXRhaW5lZCc6IG93bmVkLCAncHJvYmVzJzogW119CiAgICB0cnk6CiAgICAgICAgaWYgcGxhdGZvcm0uc3lzdGVtKCkgIT0gJ0xpbnV4JzogcmFpc2UgVW5hdmFpbGFibGUoJ0xJTlVYX1JFUVVJUkVEJykKICAgICAgICBjYW5kaWRhdGVzID0gbG9jYXRpb25zKFBhdGgoJy9wcm9jL3NlbGYvbW91bnRpbmZvJykucmVhZF90ZXh0KCksIFBhdGgoJy9wcm9jL3NlbGYvY2dyb3VwJykucmVhZF90ZXh0KCkpCiAgICAgICAgbWF0Y2hlcyA9IFt4IGZvciB4IGluIGNhbmRpZGF0ZXMgaWYgc3RyKG9zLmdldHBpZCgpKSBpbiBzdHIocmVhZChQYXRoKHhbJ2N1cnJlbnQnXSkvJ2Nncm91cC5wcm9jcycpKS5zcGxpdGxpbmVzKCldCiAgICAgICAgaWYgbGVuKG1hdGNoZXMpICE9IDE6IHJhaXNlIFVuYXZhaWxhYmxlKCdDVVJSRU5UX0NHUk9VUF9QSURfTUFQUElOR19OT1RfVU5JUVVFJykKICAgICAgICBsb2NhdGlvbiA9IG1hdGNoZXNbMF07IHBhcmVudCA9IFBhdGgobG9jYXRpb25bJ2N1cnJlbnQnXSk7IG1vdW50ID0gUGF0aChsb2NhdGlvblsnbW91bnQnXSkKICAgICAgICBpZiBwYXJlbnQuaXNfc3ltbGluaygpIG9yIHBhcmVudC5yZXNvbHZlKCkgIT0gcGFyZW50IG9yIG5vdCBwYXJlbnQuaXNfcmVsYXRpdmVfdG8obW91bnQpOgogICAgICAgICAgICByYWlzZSBVbmF2YWlsYWJsZSgnQ1VSUkVOVF9DR1JPVVBfUEFUSF9OT1RfQ0FOT05JQ0FMJykKICAgICAgICByZXBvcnQudXBkYXRlKHBsYXRmb3JtPXBsYXRmb3JtLnBsYXRmb3JtKCksIHB5dGhvbj1wbGF0Zm9ybS5weXRob25fdmVyc2lvbigpLCB1aWQ9b3MuZ2V0ZXVpZCgpLAogICAgICAgICAgICAgICAgICAgICAgYWZmaW5pdHk9c29ydGVkKG9zLnNjaGVkX2dldGFmZmluaXR5KDApKSwgY2dyb3VwTG9jYXRpb249bG9jYXRpb24pCiAgICAgICAgYW5jZXN0b3JzID0gW107IGN1cnJlbnQgPSBwYXJlbnQKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBhbmNlc3RvcnMuYXBwZW5kKHsncGF0aCc6IHN0cihjdXJyZW50KSwgJ2NvbnRyb2xzJzoge246IHJlYWQoY3VycmVudC9uKSBmb3IgbiBpbiBDT05UUk9MU319KQogICAgICAgICAgICBpZiBjdXJyZW50ID09IG1vdW50OiBicmVhawogICAgICAgICAgICBpZiBjdXJyZW50LnBhcmVudCA9PSBjdXJyZW50IG9yIG5vdCBjdXJyZW50LnBhcmVudC5pc19yZWxhdGl2ZV90byhtb3VudCk6CiAgICAgICAgICAgICAgICByYWlzZSBVbmF2YWlsYWJsZSgnQU5DRVNUT1JfV0FMS19JTkNPTVBMRVRFJykKICAgICAgICAgICAgY3VycmVudCA9IGN1cnJlbnQucGFyZW50CiAgICAgICAgcmVwb3J0WydhbmNlc3RvcnMnXSA9IGFuY2VzdG9ycwogICAgICAgIHJlcG9ydFsnb2JzZXJ2ZWRFbnZlbG9wZSddID0gYnVkZ2V0X2ZpdHMoYW5jZXN0b3JzLCByZXBvcnRbJ2FmZmluaXR5J10pCiAgICAgICAgcm9vdCA9IHBhcmVudCAvICgnY2xpbmljLWFjdGl2aXR5LWVudi0nK3J1bikKICAgICAgICAjIEFuIGFjdHVhbCBta2RpciBhdHRlbXB0IGRpc3Rpbmd1aXNoZXMgRVJPRlMvRUFDQ0VTIGZyb20gb3MuYWNjZXNzIGhpbnRzLgogICAgICAgIHRyeTogcm9vdC5ta2RpcigpCiAgICAgICAgZXhjZXB0IE9TRXJyb3IgYXMgZXJyb3I6CiAgICAgICAgICAgIHJlcG9ydFsnb3duZWRDaGlsZENyZWF0aW9uRXJybm8nXSA9IGVycm9yLmVycm5vCiAgICAgICAgICAgIHJhaXNlIFVuYXZhaWxhYmxlKCdDR1JPVVBfUkVBRF9PTkxZJyBpZiBlcnJvci5lcnJubyA9PSBlcnJuby5FUk9GUyBlbHNlICdDR1JPVVBfQ0hJTERfQ1JFQVRJT05fREVOSUVEJykKICAgICAgICBvd25lZC5hcHBlbmQoc3RyKHJvb3QpKQogICAgICAgIGlmIG5vdCB7J2NwdScsICdtZW1vcnknfSA8PSBzZXQoc3RyKHJlYWQocGFyZW50LydjZ3JvdXAuc3VidHJlZV9jb250cm9sJykpLnNwbGl0KCkpOgogICAgICAgICAgICByYWlzZSBVbmF2YWlsYWJsZSgnQ1BVX01FTU9SWV9OT1RfREVMRUdBVEVEX1RPX0NISUxEUkVOJykKICAgICAgICBpZiByZWFkKHJvb3QvJ2Nncm91cC5wcm9jcycpICE9ICcnOiByYWlzZSBVbmF2YWlsYWJsZSgnTkVXX0NHUk9VUF9OT1RfRU1QVFknKQogICAgICAgIChyb290LydjZ3JvdXAuc3VidHJlZV9jb250cm9sJykud3JpdGVfdGV4dCgnK2NwdSArbWVtb3J5JykKICAgICAgICBsZWF2ZXMgPSB7fQogICAgICAgIGxpbWl0cyA9IHsnc3RhY2snOiAoQlVER0VUWydzdGFjayddWydjcHVNYXgnXSwgOCoxMDI0KiozKSwKICAgICAgICAgICAgICAgICAgJ2dlbmVyYXRvcic6IChCVURHRVRbJ2dlbmVyYXRvciddWydjcHVNYXgnXSwgMioxMDI0KiozKSwKICAgICAgICAgICAgICAgICAgJ2NwdS1wcm9iZSc6ICgnMTAwMDAgMTAwMDAwJywgNjQqMTAyNCoqMiksICdtZW1vcnktcHJvYmUnOiAoJzEwMDAwMCAxMDAwMDAnLCAzMioxMDI0KioyKX0KICAgICAgICBmb3IgbmFtZSwgKGNwdSwgbWVtb3J5KSBpbiBsaW1pdHMuaXRlbXMoKToKICAgICAgICAgICAgbGVhZiA9IHJvb3QvbmFtZTsgbGVhZi5ta2RpcigpOyBvd25lZC5hcHBlbmQoc3RyKGxlYWYpKTsgbGVhdmVzW25hbWVdID0gbGVhZgogICAgICAgICAgICAobGVhZi8nY3B1Lm1heCcpLndyaXRlX3RleHQoY3B1KTsgKGxlYWYvJ21lbW9yeS5tYXgnKS53cml0ZV90ZXh0KHN0cihtZW1vcnkpKQogICAgICAgICAgICAobGVhZi8nbWVtb3J5LnN3YXAubWF4Jykud3JpdGVfdGV4dCgnMCcpCiAgICAgICAgICAgIGlmIHJlYWQobGVhZi8nY3B1Lm1heCcpICE9IGNwdSBvciByZWFkKGxlYWYvJ21lbW9yeS5tYXgnKSAhPSBzdHIobWVtb3J5KSBvciByZWFkKGxlYWYvJ21lbW9yeS5zd2FwLm1heCcpICE9ICcwJzoKICAgICAgICAgICAgICAgIHJhaXNlIFVuYXZhaWxhYmxlKCdDT05UUk9MTEVSX1JFQURCQUNLX01JU01BVENIJykKICAgICAgICByZXBvcnRbJ2NvbmZpZ3VyZWRHcm91cHMnXSA9IHtuOiB7a2V5OiByZWFkKHAva2V5KSBmb3Iga2V5IGluIFsnY3B1Lm1heCcsJ21lbW9yeS5tYXgnLCdtZW1vcnkuc3dhcC5tYXgnXX0gZm9yIG4scCBpbiBsZWF2ZXMuaXRlbXMoKX0KCiAgICAgICAgZGVmIGNoaWxkKG5hbWUsIGtpbmQpOgogICAgICAgICAgICBsZWFmID0gbGVhdmVzW25hbWVdCiAgICAgICAgICAgIHByb2Nlc3MgPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgJy1JJywgJy1TJywgJy1jJywgSEVMUEVSLCBraW5kXSwKICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwgc3RkZXJyPXN1YnByb2Nlc3MuREVWTlVMTCwKICAgICAgICAgICAgICAgIGVudj17J1BBVEgnOiBvcy5kZWZwYXRoLCAnTEFORyc6ICdDLlVURi04J30sIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQocHJvY2VzcykKICAgICAgICAgICAgaWYgcHJvY2Vzcy5wb2xsKCkgaXMgbm90IE5vbmU6IHJhaXNlIFVuYXZhaWxhYmxlKCdIRUxQRVJfRVhJVEVEX0JFRk9SRV9BVFRBQ0hNRU5UJykKICAgICAgICAgICAgKGxlYWYvJ2Nncm91cC5wcm9jcycpLndyaXRlX3RleHQoc3RyKHByb2Nlc3MucGlkKSkKICAgICAgICAgICAgaWYgc3RyKHByb2Nlc3MucGlkKSBub3QgaW4gc3RyKHJlYWQobGVhZi8nY2dyb3VwLnByb2NzJykpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgICAgIHJhaXNlIFVuYXZhaWxhYmxlKCdIRUxQRVJfTk9UX0lOX09XTkVEX0dST1VQJykKICAgICAgICAgICAgbWVtYmVyc2hpcCA9IHJlYWQoUGF0aCgnL3Byb2MnKS9zdHIocHJvY2Vzcy5waWQpLydjZ3JvdXAnKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShtZW1iZXJzaGlwLHN0cikgb3IgKCdjbGluaWMtYWN0aXZpdHktZW52LScrcnVuKycvJytuYW1lKSBub3QgaW4gbWVtYmVyc2hpcDoKICAgICAgICAgICAgICAgIHJhaXNlIFVuYXZhaWxhYmxlKCdIRUxQRVJfTUVNQkVSU0hJUF9NSVNNQVRDSCcpCiAgICAgICAgICAgIG91dHB1dCwgXyA9IHByb2Nlc3MuY29tbXVuaWNhdGUoYidydW5cbicsIHRpbWVvdXQ9OCkKICAgICAgICAgICAgcmVwb3J0Wydwcm9iZXMnXS5hcHBlbmQoeydncm91cCc6IG5hbWUsICdwaWQnOiBwcm9jZXNzLnBpZCwgJ21lbWJlcnNoaXAnOiBtZW1iZXJzaGlwLCAnZXhpdENvZGUnOiBwcm9jZXNzLnJldHVybmNvZGV9KQogICAgICAgICAgICByZXR1cm4gcHJvY2Vzcy5yZXR1cm5jb2RlLCBvdXRwdXQKCiAgICAgICAgZm9yIG5hbWUgaW4gWydzdGFjaycsJ2dlbmVyYXRvciddOgogICAgICAgICAgICBjb2RlLCBvdXRwdXQgPSBjaGlsZChuYW1lLCAnaWRlbnRpdHknKQogICAgICAgICAgICBpZiBjb2RlICE9IDAgb3IgJ21lbWJlcnNoaXAnIG5vdCBpbiBqc29uLmxvYWRzKG91dHB1dCk6IHJhaXNlIFVuYXZhaWxhYmxlKCdCVURHRVRfR1JPVVBfQVRUQUNITUVOVF9GQUlMRUQnKQogICAgICAgIGJlZm9yZSA9IGNvdW50ZXJzKGxlYXZlc1snY3B1LXByb2JlJ10vJ2NwdS5zdGF0Jyk7IGNvZGUsIF8gPSBjaGlsZCgnY3B1LXByb2JlJywnY3B1JykKICAgICAgICBhZnRlciA9IGNvdW50ZXJzKGxlYXZlc1snY3B1LXByb2JlJ10vJ2NwdS5zdGF0JykKICAgICAgICBpZiBjb2RlICE9IDAgb3IgYWZ0ZXIuZ2V0KCducl90aHJvdHRsZWQnLDApIDw9IGJlZm9yZS5nZXQoJ25yX3Rocm90dGxlZCcsMCk6CiAgICAgICAgICAgIHJhaXNlIFVuYXZhaWxhYmxlKCdDUFVfVEhST1RUTElOR19OT1RfUFJPVkVOJykKICAgICAgICByZXBvcnRbJ2NwdVRocm90dGxpbmdPYnNlcnZlZCddID0gVHJ1ZQogICAgICAgIGJlZm9yZSA9IGNvdW50ZXJzKGxlYXZlc1snbWVtb3J5LXByb2JlJ10vJ21lbW9yeS5ldmVudHMnKTsgY29kZSwgXyA9IGNoaWxkKCdtZW1vcnktcHJvYmUnLCdtZW1vcnknKQogICAgICAgIGFmdGVyID0gY291bnRlcnMobGVhdmVzWydtZW1vcnktcHJvYmUnXS8nbWVtb3J5LmV2ZW50cycpCiAgICAgICAgaWYgY29kZSAhPSAtc2lnbmFsLlNJR0tJTEwgb3IgYWZ0ZXIuZ2V0KCdvb21fa2lsbCcsMCkgPD0gYmVmb3JlLmdldCgnb29tX2tpbGwnLDApOgogICAgICAgICAgICByYWlzZSBVbmF2YWlsYWJsZSgnQ09ORklORURfTUVNT1JZX0xJTUlUX05PVF9QUk9WRU4nKQogICAgICAgIHJlcG9ydFsnY29uZmluZWRIZWxwZXJPb21PYnNlcnZlZCddID0gVHJ1ZQogICAgICAgIHJlcG9ydC51cGRhdGUoc3RhdHVzPSdDQVBBQklMSVRZX1BBU1NfRlVMTF9UUklBTF9QRU5ESU5HJywgcmVzb3VyY2VFbmZvcmNlbWVudD0nU1lOVEhFVElDX0hFTFBFUlNfT05MWScsCiAgICAgICAgICAgICAgICAgICAgICBuZXh0QWN0aW9uPSdSVU5fR1VBUkRFRF9GVUxMX1NUQUNLX1BST0ZJTEVfVU5ERVJfUFJPVkVOX0dST1VQU19CRUZPUkVfRDBfUEFTUycpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOgogICAgICAgIHJlcG9ydC51cGRhdGUoc3RhdHVzPSdDQVBBQklMSVRZX1VOQVZBSUxBQkxFJyBpZiBpc2luc3RhbmNlKGVycm9yLFVuYXZhaWxhYmxlKSBlbHNlICdQUk9CRV9GQUlMRUQnLAogICAgICAgICAgICAgICAgICAgICAgcmVhc29uPXN0cihlcnJvcikgaWYgaXNpbnN0YW5jZShlcnJvcixVbmF2YWlsYWJsZSkgZWxzZSB0eXBlKGVycm9yKS5fX25hbWVfXywKICAgICAgICAgICAgICAgICAgICAgIG5leHRBY3Rpb249J0RPX05PVF9CT09UU1RSQVBfT1JfQkVOQ0hNQVJLOyBQUk9WSURFX0FfVFJJQUxfV0lUSF9SRVFVSVJFRF9ERUxFR0FUSU9OJykKICAgICAgICBpZiBpc2luc3RhbmNlKGVycm9yLE9TRXJyb3IpOiByZXBvcnRbJ2Vycm5vJ10gPSBlcnJvci5lcnJubwogICAgZmluYWxseToKICAgICAgICBjbGVhbnVwX2Vycm9ycyA9IFtdCiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBpZiBwcm9jZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHRyeTogb3Mua2lsbHBnKHByb2Nlc3MucGlkLCBzaWduYWwuU0lHVEVSTSkKICAgICAgICAgICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOiBwYXNzCiAgICAgICAgICAgICAgICAgICAgdHJ5OiBwcm9jZXNzLndhaXQodGltZW91dD0yKQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgICAgICAgICAgICAgICAgICB0cnk6IG9zLmtpbGxwZyhwcm9jZXNzLnBpZCwgc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBQcm9jZXNzTG9va3VwRXJyb3I6IHBhc3MKICAgICAgICAgICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9MikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjogY2xlYW51cF9lcnJvcnMuYXBwZW5kKHR5cGUoZXJyb3IpLl9fbmFtZV9fKQogICAgICAgIHJlcG9ydFsnb3duZWRIZWxwZXJQcm9jZXNzZXNTdG9wcGVkJ10gPSBhbGwocC5wb2xsKCkgaXMgbm90IE5vbmUgZm9yIHAgaW4gcHJvY2Vzc2VzKQogICAgICAgIGlmIGNsZWFudXBfZXJyb3JzOgogICAgICAgICAgICByZXBvcnQudXBkYXRlKHN0YXR1cz0nUFJPQkVfRkFJTEVEJywgcmVhc29uPSdPV05FRF9IRUxQRVJfQ0xFQU5VUF9GQUlMRUQnLCBjbGVhbnVwRXJyb3JzPWNsZWFudXBfZXJyb3JzKQogICAgICAgIHJlcG9ydFsnZmluaXNoZWRBdFV0YyddID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCkKICAgIHJldHVybiByZXBvcnQKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcHJpbnQoanNvbi5kdW1wcyhydW5fcHJvYmUoKSwgaW5kZW50PTIpKQo='
source=base64.b64decode(PROBE_BASE64,validate=True)
assert hashlib.sha256(source).hexdigest()==PROBE_SHA256
scope={'__name__':'activity_environment_probe'}
exec(compile(source,'<verified-d0-probe>','exec'),scope)
report=scope['run_probe']()
report['probeSourceSha256']=PROBE_SHA256
output=Path('/kaggle/working')/('clinic-activity-d0-env-'+report['run'])
output.mkdir(mode=0o700)
raw=(json.dumps(report,indent=2)+'\n').encode()
with (output/'environment.json').open('xb') as f:f.write(raw)
manifest={'format':'clinic-activity-d0-probe-result-v1','probeSourceSha256':PROBE_SHA256,
          'files':[{'path':'environment.json','bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()}]}
with (output/'manifest.json').open('x') as f:json.dump(manifest,f,indent=2)
archive=output.with_suffix('.zip')
with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED) as z:
    z.write(output/'environment.json','environment.json');z.write(output/'manifest.json','manifest.json')
print(json.dumps({k:report[k] for k in ['status','d0','resourceEnforcement','ownedHelperProcessesStopped']},indent=2))
if 'reason' in report:print('reason:',report['reason'])
print('No application startup, database connection, migration or benchmark. Send this ZIP for the environment decision.')
display(FileLink(str(archive)))
